# 01 插值问题概览

插值从离散数据点出发：

$$
(x_0,y_0), (x_1,y_1), \dots, (x_n,y_n),
$$

目标是构造一个函数 $p(x)$，使得

$$
p(x_i)=y_i,\qquad i=0,1,\dots,n.
$$

本节先建立问题背景，比较“插值”和“拟合”的区别，并说明为什么节点分布会在算法选择之前就影响数值稳定性。


## 插值、拟合与数据可信度

插值要求曲线精确通过所有节点。这适合查表值、精确函数采样或高精度模拟数据。拟合则允许残差，更适合带噪声的实验测量。

因此，数值计算中的问题不是简单地“画一条曲线”，而是要先判断：

* 数据值是否可信；
* 插值函数属于什么函数空间；
* 节点分布是否会造成不稳定；
* 当前任务更需要全局光滑性还是局部稳健性。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import lagrange_interpolate, piecewise_linear_interpolate

x = np.array([0.0, 0.7, 1.5, 2.4, 3.0, 4.0])
y = np.sin(x) + 0.15 * x
x_eval = np.linspace(x.min(), x.max(), 400)

poly = lagrange_interpolate(x, y, x_eval)
linear = piecewise_linear_interpolate(x, y, x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, poly, label="全局多项式插值")
plt.plot(x_eval, linear, label="分段线性插值")
plt.scatter(x, y, color="black", zorder=3, label="数据点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("同一组数据的两种插值结果")
plt.legend()
plt.tight_layout()
plt.show()


## 插值不等于最小二乘拟合

如果数据含有噪声，强制曲线通过每一个点可能会让插值函数跟着噪声振荡。低次最小二乘拟合可能忽略部分局部细节，但往往能更好地描述整体趋势。


In [ ]:
rng = np.random.default_rng(2026)
x_noisy = np.linspace(0.0, 4.0, 9)
y_clean = np.sin(x_noisy) + 0.15 * x_noisy
y_noisy = y_clean + 0.12 * rng.normal(size=x_noisy.size)

x_plot = np.linspace(x_noisy.min(), x_noisy.max(), 500)
interpolant = lagrange_interpolate(x_noisy, y_noisy, x_plot)
fit_coefficients = np.polyfit(x_noisy, y_noisy, deg=3)
least_squares_fit = np.polyval(fit_coefficients, x_plot)

plt.figure(figsize=(8, 4.5))
plt.plot(x_plot, interpolant, label="8 次插值多项式")
plt.plot(x_plot, least_squares_fit, label="3 次最小二乘拟合")
plt.scatter(x_noisy, y_noisy, color="black", zorder=3, label="含噪声数据")
plt.xlabel("x")
plt.ylabel("函数值")
plt.title("精确插值可能放大噪声")
plt.legend()
plt.tight_layout()
plt.show()


## 插值空间与唯一性

对 $n+1$ 个互异节点，在 $\mathcal P_n$ 中存在唯一的插值多项式。计算上可以通过 Vandermonde 矩阵观察这个结论。矩阵可逆说明解存在且唯一，但条件数很大时，数值计算仍可能不稳定。


In [ ]:
def vandermonde_condition(nodes):
    matrix = np.vander(nodes, N=nodes.size, increasing=True)
    return np.linalg.cond(matrix)

for n in [5, 9, 13]:
    equal_nodes = np.linspace(-1.0, 1.0, n)
    print(f"节点数={n:2d}, Vandermonde 条件数={vandermonde_condition(equal_nodes):10.3e}")


## 小结

* 插值精确满足节点值。
* 对含噪声数据，拟合往往比插值更稳健。
* 互异节点上的多项式插值具有唯一性，但唯一性不等于稳定性。
* 下一节将系统讨论全局多项式插值。
